# Java Bug Fixing Model Training
This notebook trains the multi-task transformer model on Kaggle's GPUs

## 1. Setup Environment

In [ ]:
# Install requirements
!pip install -q transformers==4.40.0 datasets==2.19.0 torch==2.3.0 accelerate

# Clone project repository
!git clone -q https://github.com/yourusername/BugFixer.git
%cd BugFixer

# Install project dependencies
!pip install -q -r requirements.txt

# Setup environment
import sys
sys.path.append('/kaggle/working/BugFixer')

## 2. Load Processed Dataset

In [ ]:
from datasets import load_from_disk
import matplotlib.pyplot as plt

# Load tokenized dataset
dataset_path = "/kaggle/input/processed-java-refinement"  # Add your dataset first
tokenized_dataset = load_from_disk(dataset_path)

# Show dataset structure
print("Dataset features:", tokenized_dataset["train"].features)
print("Train samples:", len(tokenized_dataset["train"]))
print("Validation samples:", len(tokenized_dataset["validation"]))
print("Test samples:", len(tokenized_dataset["test"]))

# Show a sample
sample = tokenized_dataset["train"][0]
print("\nSample input IDs:", sample["input_ids"][:10], "...")
print("Sample attention mask:", sample["attention_mask"][:10], "...")
print("Sample labels:", sample["labels"][:10], "...")
print("Error label:", sample["error_label"])

## 3. Initialize Model

In [ ]:
import torch
from src.model.architecture import MultiTaskCodeT5

# Configuration
MODEL_NAME = "Salesforce/codet5-base-java"
NUM_CLASSES = 4  # Syntax, Logical, Runtime, Other

# Initialize model
model = MultiTaskCodeT5(model_name=MODEL_NAME, num_classes=NUM_CLASSES)

# Log model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model structure:")
print(model)

## 4. Create Data Loaders

In [ ]:
from torch.utils.data import Dataset, DataLoader

class JavaRefinementDataset(Dataset):
    def __init__(self, dataset, split="train"):
        self.data = dataset[split]
        self.split = split
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "labels": torch.tensor(item["labels"]),
            "error_label": torch.tensor(item["error_label"])
        }

# Create datasets
train_dataset = JavaRefinementDataset(tokenized_dataset, "train")
val_dataset = JavaRefinementDataset(tokenized_dataset, "validation")

# Create data loaders
BATCH_SIZE = 32  # Use larger batch size on Kaggle GPU
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=2
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=2
)

# Show batch info
batch = next(iter(train_loader))
print("\nBatch keys:", batch.keys())
print("Input IDs shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)
print("Error labels shape:", batch["error_label"].shape)

## 5. Configure Training

In [ ]:
from transformers import TrainingArguments, Trainer
from src.model.train_kaggle import MultiTaskTrainer

# Training configuration
EPOCHS = 10
LEARNING_RATE = 5e-5
WARMUP_STEPS = 500
LOGGING_STEPS = 100
EVAL_STEPS = 500
SAVE_STEPS = 1000
OUTPUT_DIR = "/kaggle/working/output"

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=LOGGING_STEPS,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    fp16=True,
    gradient_accumulation_steps=1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    prediction_loss_only=True,
    dataloader_num_workers=4,
    ddp_find_unused_parameters=False,
)

# Initialize trainer
trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Training configuration complete!")

## 6. Train Model

In [ ]:
# Start training
print("Starting training...")
train_result = trainer.train()

# Save final model
final_model_path = f"{OUTPUT_DIR}/final_model"
trainer.save_model(final_model_path)
print(f"Saved final model to {final_model_path}")

# Evaluate on validation set
print("Evaluating on validation set...")
eval_metrics = trainer.evaluate()
print("Validation metrics:", eval_metrics)

# Save best model
best_model_path = f"{OUTPUT_DIR}/best_model"
trainer.save_model(best_model_path)
print(f"Saved best model to {best_model_path}")

print("Training complete!")

## 7. Training Analysis

In [ ]:
import json
import matplotlib.pyplot as plt

# Load training logs
with open(f"{OUTPUT_DIR}/trainer_state.json", "r") as f:
    logs = json.load(f)

# Extract training metrics
train_loss = [log["loss"] for log in logs["log_history"] if "loss" in log]
eval_loss = [log["eval_loss"] for log in logs["log_history"] if "eval_loss" in log]
steps = [log["step"] for log in logs["log_history"] if "loss" in log]

# Plot training curve
plt.figure(figsize=(12, 6))
plt.plot(steps, train_loss, label="Training Loss")
plt.plot(steps, eval_loss, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/training_curve.png")
plt.show()

# Print final metrics
print("Final Training Metrics:")
print(f"Training Loss: {train_loss[-1]:.4f}")
print(f"Validation Loss: {eval_loss[-1]:.4f}")
print(f"Best Validation Loss: {min(eval_loss):.4f} at step {steps[eval_loss.index(min(eval_loss))]}")

## 8. Save and Export Model

In [ ]:
# Create ZIP archive
!zip -r /kaggle/working/model.zip /kaggle/working/output

# Download link
from IPython.display import FileLink
FileLink(r'model.zip')

print("Model exported successfully!")

## 9. Test Inference (Optional)

In [ ]:
from src.model.inference import BugFixer

# Initialize model
fixer = BugFixer(f"{best_model_path}/pytorch_model.bin")

# Test sample
buggy_code = """
public int sum(int a, int b) {
    return a - b;  // Should be addition
}
"""

# Get prediction
result = fixer.predict(buggy_code)
print("Buggy Code:")
print(buggy_code)
print("\nFixed Code:")
print(result["fixed_code"])
print("\nError Type:", result["error_type"])
print("Confidence:", f"{result['confidence']:.2%}")
print("\nSuggestions:")
for i, suggestion in enumerate(result["refactoring_suggestions"]):
    print(f"{i+1}. {suggestion}")